# Multi-condition sparse FBA

This tutorial extends the sparse FBA workflow from `flux-balance-analysis.ipynb` to multiple conditions. We first solve standard multi-condition FBA to get each condition's optimal biomass. Then, as in single-condition sparse FBA, we constrain biomass to at least 90% of optimum and minimize the number of active reactions.

To make the effect visible, we use a toy network where each condition has a small preference for a different branch.

In [ ]:
import numpy as np
import pandas as pd

import corneto as cn
from corneto.methods import MultiSampleFBA

cn.info()

In [ ]:
G = cn.Graph()

# Uptake
G.add_edge((), "A", id="EX_A", default_lb=0, default_ub=10)

# Two alternative branches
G.add_edge("A", "B", id="R_AB", default_lb=0, default_ub=100)
G.add_edge("B", "C", id="R_BC", default_lb=0, default_ub=100)
G.add_edge("A", "D", id="R_AD", default_lb=0, default_ub=100)
G.add_edge("D", "C", id="R_DC", default_lb=0, default_ub=100)

# Shared pathway to biomass
G.add_edge("C", "E", id="R_CE", default_lb=0, default_ub=100)
G.add_edge("E", "F", id="R_EF", default_lb=0, default_ub=100)
G.add_edge("F", (), id="BIOMASS", default_lb=0, default_ub=100)

pd.DataFrame(
    {
        "edge_index": range(G.num_edges),
        "reaction_id": [G.get_attr_edge(i).get("id") for i in range(G.num_edges)],
    }
)

In [ ]:
G.plot()

In [ ]:
def make_condition(preferred_branch):
    # Main objective: maximize biomass (negative value because objectives are minimized).
    sample = {"BIOMASS": {"role": "objective", "value": -1.0}}
    # Small condition-specific preference for one branch.
    sample[preferred_branch] = {"role": "objective", "value": -0.01}
    return sample


condition_specs = {
    "condition_1": make_condition("R_BC"),
    "condition_2": make_condition("R_DC"),
}

multi_data = cn.Data.from_cdict(condition_specs)
sample_names = list(condition_specs)

## Standard multi-condition FBA

We start with regular multi-condition FBA to obtain the maximal biomass in each condition. These values are later used as lower bounds for sparse FBA.

In [ ]:
idx = {rid: next(iter(G.get_edges_by_attr("id", rid))) for rid in ["BIOMASS", "R_AB", "R_AD"]}

P_opt = MultiSampleFBA(lambda_reg=0.0).build(G, multi_data)
P_opt.solve(solver="highs")

biomass_opt = {name: float(P_opt.expr.flow[idx["BIOMASS"], i].value) for i, name in enumerate(sample_names)}

pd.DataFrame(
    {
        "condition": sample_names,
        "optimal_biomass": [biomass_opt[name] for name in sample_names],
    }
)

## Sparse FBA per condition (independent runs)

This mirrors the single-condition sparse FBA pattern from `flux-balance-analysis.ipynb`:

1. Solve standard FBA to get optimum biomass.
2. Add a biomass lower bound (90% of optimum).
3. Solve sparse FBA (`lambda_reg > 0`).

In [ ]:
lambda_sparse = 0.15
biomass_fraction = 0.90
biomass_targets = {name: biomass_fraction * biomass_opt[name] for name in sample_names}

single_sparse = {}
for name in sample_names:
    d = cn.Data.from_cdict({name: condition_specs[name]})
    P = MultiSampleFBA(lambda_reg=lambda_sparse).build(G, d)
    P += P.expr.flow[idx["BIOMASS"]] >= biomass_targets[name]
    P.solve(solver="highs")
    single_sparse[name] = P

## Coupled multi-condition sparse FBA

Now we solve both conditions together with structured sparsity. The same biomass lower bounds are enforced, but regularization is applied jointly across conditions.

In [ ]:
P_multi_sparse = MultiSampleFBA(lambda_reg=lambda_sparse).build(G, multi_data)
for i, name in enumerate(sample_names):
    P_multi_sparse += P_multi_sparse.expr.flow[idx["BIOMASS"], i] >= biomass_targets[name]
P_multi_sparse.solve(solver="highs")

## Compare independent vs coupled sparse solutions

In [ ]:
edge_ids = [G.get_attr_edge(i).get("id") for i in range(G.num_edges)]


def active_ids(problem, condition=None, tol=1e-6):
    flow = np.asarray(problem.expr.flow.value)
    if condition is not None:
        flow = flow[:, condition]
    return {edge_ids[i] for i in np.where(np.abs(flow) > tol)[0]}


def branch_label(ab_flux, ad_flux, tol=1e-6):
    ab_on = abs(float(ab_flux)) > tol
    ad_on = abs(float(ad_flux)) > tol
    if ab_on and not ad_on:
        return "branch_1 (A->B->C)"
    if ad_on and not ab_on:
        return "branch_2 (A->D->C)"
    if ab_on and ad_on:
        return "both"
    return "none"


rows = []
for i, name in enumerate(sample_names):
    P_single = single_sparse[name]
    rows.append(
        {
            "setting": f"single sparse ({name})",
            "selected_branch": branch_label(
                P_single.expr.flow[idx["R_AB"]].value, P_single.expr.flow[idx["R_AD"]].value
            ),
            "biomass": float(P_single.expr.flow[idx["BIOMASS"]].value),
            "active_reactions": len(active_ids(P_single)),
        }
    )
    rows.append(
        {
            "setting": f"multi sparse coupled ({name})",
            "selected_branch": branch_label(
                P_multi_sparse.expr.flow[idx["R_AB"], i].value, P_multi_sparse.expr.flow[idx["R_AD"], i].value
            ),
            "biomass": float(P_multi_sparse.expr.flow[idx["BIOMASS"], i].value),
            "active_reactions": len(active_ids(P_multi_sparse, condition=i)),
        }
    )

comparison = pd.DataFrame(rows)
comparison

In [ ]:
single_union = set().union(*(active_ids(single_sparse[name]) for name in sample_names))
multi_union = set().union(*(active_ids(P_multi_sparse, condition=i) for i in range(len(sample_names))))

summary = pd.DataFrame(
    [
        {
            "single_sparse_union": len(single_union),
            "multi_sparse_union": len(multi_union),
            "union_reduction": len(single_union) - len(multi_union),
        }
    ]
)

# Expected in this setup: coupled sparse FBA uses fewer reactions in the union across conditions.
assert len(multi_union) < len(single_union)

summary

In [ ]:
pd.DataFrame(
    {
        "single_sparse_union": pd.Series(sorted(single_union)),
        "multi_sparse_union": pd.Series(sorted(multi_union)),
    }
)

In [ ]:
active_multi_c1 = np.abs(P_multi_sparse.expr.flow.value[:, 0]) > 1e-6
active_multi_c2 = np.abs(P_multi_sparse.expr.flow.value[:, 1]) > 1e-6

G.edge_subgraph(active_multi_c1).plot(graph_attr={"rankdir": "LR", "center": "1"})

In [ ]:
G.edge_subgraph(active_multi_c2).plot(graph_attr={"rankdir": "LR", "center": "1"})

In this toy example, independent sparse runs choose different branches, while the coupled multi-condition sparse run aligns branch usage and reduces the union of active reactions across conditions.